In [1]:
from ifc.config import load_data_config, load_split_config, resolve_project_root
from pathlib import Path
import pandas as pd

data_cfg = load_data_config() 
split_cfg = load_split_config()  
train_path, test_path = data_cfg.resolve_paths()    
print(data_cfg,"\n")

print(f'training path:{train_path}, exists? {train_path.exists()}')
print(f'tets path:{test_path}, exists? {test_path.exists()}')

id_col, time_col, target = data_cfg.id_col, data_cfg.time_col, data_cfg.target_col

print(f"Company_id column: {id_col}\nFiscal_year column: {time_col}\nTarget column: {target}")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("train_df:", train_df.shape)
print("test_df :", test_df.shape)


DataConfig(train_path=WindowsPath('data/processed/train_data.csv'), test_path=WindowsPath('data/processed/test_features.csv'), id_col='company_id', time_col='fiscal_year', target_col='revenue_change', drop_cols=['bankruptcy_next_year', 'financial_health_class']) 

training path:C:\Users\morio\OneDrive\Desktop\Projects\italian-financial-challenge\data\processed\train_data.csv, exists? True
tets path:C:\Users\morio\OneDrive\Desktop\Projects\italian-financial-challenge\data\processed\test_features.csv, exists? True
Company_id column: company_id
Fiscal_year column: fiscal_year
Target column: revenue_change
train_df: (11828, 30)
test_df : (5811, 27)


In [2]:
train_df[[time_col,target]].describe().T

,count,mean,std,min,25%,50%,75%,max
fiscal_year,11828.0,2019.49535,1.116482,2018.00,2018.00,2019.00,2020.00,2021.00
revenue_change,8829.0,453.43457,4601.920625,-99.94,-68.59,3.04,238.85,302126.48


In [3]:
test_df[time_col].describe().T

count    5811.000000
mean     2022.498193
std         0.500040
min      2022.000000
25%      2022.000000
50%      2022.000000
75%      2023.000000
max      2023.000000
Name: fiscal_year, dtype: float64

In [4]:
unique_id = train_df[id_col].nunique()
unique_id_percentage = unique_id/len(train_df[id_col])

print(f'unique IDs:{unique_id}')
print(f'unique id percentage:{unique_id_percentage}')

unique IDs:2999
unique id percentage:0.25355089617855936


In [5]:
def missing_summary(df):
    missing_values = df.isnull().sum()
    missing_pct = (missing_values / len(df)) * 100

    missing_df = pd.DataFrame({
        'Missing Count': missing_values,
        'Percentage': missing_pct
    })
    missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
    return missing_df




In [6]:
missing_train = missing_summary(train_df)
print("Missing Values train:")
print(missing_train)

Missing Values train:
                Missing Count  Percentage
revenue_change           2999   25.355090
province                  919    7.769699
roe                        45    0.380453
leverage                   45    0.380453


In [7]:
missing_test = missing_summary(test_df)
print("Missing Values test")
print(missing_test)

Missing Values test
          Missing Count  Percentage
province            451    7.761143
roe                  14    0.240922
leverage             14    0.240922


In [8]:
key = [id_col, time_col]

dup_train = train_df.duplicated(subset=key).sum()
print("dup_train:", dup_train)

dup_test = test_df.duplicated(subset=key).sum()
print("dup_test:", dup_test)

if dup_train > 0:
    display(train_df.loc[train_df.duplicated(subset=key, keep=False), key].sort_values(key).head(20))

if dup_test > 0:
    display(test_df.loc[test_df.duplicated(subset=key, keep=False), key].sort_values(key).head(20))


dup_train: 0
dup_test: 0


In [9]:

df = train_df.copy()

first_year = df.groupby("company_id")["fiscal_year"].min()
df = df.join(first_year.rename("first_year"), on="company_id")

missing_in_first = df.loc[df["revenue_change"].isna() & (df["fiscal_year"] == df["first_year"])].shape[0]
missing_total = df["revenue_change"].isna().sum()

missing_not_first = df.loc[df["revenue_change"].isna() & (df["fiscal_year"] != df["first_year"])].shape[0]

print("missing_total:", missing_total)
print("missing_in_first_year:", missing_in_first)
print("missing_not_first_year (anomalies):", missing_not_first)

if missing_not_first > 0:
    display(df.loc[df["revenue_change"].isna() & (df["fiscal_year"] != df["first_year"]),
                   ["company_id","fiscal_year","first_year","production_value"]].head(20))


missing_total: 2999
missing_in_first_year: 2999
missing_not_first_year (anomalies): 0


In [10]:
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)

only_in_train = sorted(list(train_cols - test_cols))
only_in_test = sorted(list(test_cols - train_cols))

print("Only in train:", only_in_train)
print("Only in test :", only_in_test)


Only in train: ['bankruptcy_next_year', 'financial_health_class', 'revenue_change']
Only in test : []


In [11]:
years = sorted(train_df[time_col].unique())
print("years_in_train_df:", years)

# counts per year
print(train_df[time_col].value_counts().sort_index())


years_in_train_df: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
fiscal_year
2018    2961
2019    2979
2020    2956
2021    2932
Name: count, dtype: int64


# Data split

In [12]:
from ifc.split import apply_forecasting_holdout_split, build_forecasting_test_frame

df_train, df_val = apply_forecasting_holdout_split(train_df, split_cfg, drop_cols=data_cfg.drop_cols)
df_test_forecast = build_forecasting_test_frame(train_df, test_df, split_cfg, drop_cols=data_cfg.drop_cols)

print("df_train_forecast:", df_train.shape, "years:", sorted(df_train[time_col].unique()))
print("df_val_forecast  :", df_val.shape, "years:", sorted(df_val[time_col].unique()))
print("df_test_forecast :", df_test_forecast.shape, "years:", sorted(df_test_forecast[time_col].unique()))

df_train_forecast: (5897, 28) years: [np.int64(2019), np.int64(2020)]
df_val_forecast  : (2932, 28) years: [np.int64(2021)]
df_test_forecast : (5811, 27) years: [np.int64(2022), np.int64(2023)]


In [14]:
df_train.head()


,company_id,fiscal_year,revenue_change,prev_province,prev_region,prev_ateco_sector,prev_legal_form,prev_years_in_business,prev_total_fixed_assets,prev_current_assets,...,prev_financial_income,prev_financial_expenses,prev_net_profit_loss,prev_roe,prev_roi,prev_leverage,prev_current_ratio,prev_quick_ratio,prev_debt_to_assets,prev_profit_margin
1,COMP_00000,2019,-76.77,NaN,Campania,47.0,SRL,32.0,2.422343e+08,9.504819e+08,...,4886785.12,20068071.29,2.152049e+08,0.5772,0.1973,2.1987,1.6967,1.0180,0.6874,0.1165
2,COMP_00000,2020,1521.81,NaN,Campania,47.0,SRL,33.0,3.983639e+07,1.411226e+08,...,58246.06,2537224.20,3.677495e+07,0.6451,0.2172,2.1743,1.9778,1.1867,0.6850,0.0857
5,COMP_00001,2019,287.89,CA,Sardegna,62.0,SRL,8.0,8.348035e+07,1.325697e+08,...,640538.94,5860639.22,1.423603e+07,0.2168,0.0930,2.2900,1.7484,1.0490,0.6960,0.0346
6,COMP_00001,2020,-88.55,CA,Sardegna,62.0,SRL,9.0,3.073529e+08,6.198199e+08,...,866498.44,19304798.48,4.520087e+07,0.2293,0.0696,3.7038,1.8612,1.1167,0.7874,0.0283
9,COMP_00002,2019,1865.41,BA,Puglia,41.0,SAS,35.0,3.906155e+07,2.581125e+07,...,9342.13,978717.47,4.222250e+06,0.1388,0.0802,1.1332,1.2336,0.7401,0.5312,0.0857


In [16]:
df_train.describe().T

,count,mean,std,min,25%,50%,75%,max
fiscal_year,5897.0,2.019501e+03,5.000408e-01,2.019000e+03,2.019000e+03,2.020000e+03,2.020000e+03,2.020000e+03
revenue_change,5897.0,4.358499e+02,3.474444e+03,-9.994000e+01,-6.920000e+01,1.100000e+00,2.381900e+02,2.130303e+05
prev_ateco_sector,5897.0,4.583856e+01,1.669567e+01,1.000000e+01,4.100000e+01,4.600000e+01,5.600000e+01,8.200000e+01
prev_years_in_business,5897.0,3.429625e+01,1.982691e+01,0.000000e+00,1.700000e+01,3.400000e+01,5.100000e+01,6.900000e+01
prev_total_fixed_assets,5897.0,9.674398e+08,5.112752e+09,1.656874e+06,7.314590e+07,1.886777e+08,5.185635e+08,1.505032e+11
prev_current_assets,5897.0,1.395298e+09,6.747800e+09,3.983805e+06,1.240835e+08,3.043228e+08,8.119364e+08,2.432396e+11
prev_total_assets,5897.0,2.362738e+09,1.130449e+10,5.640679e+06,2.085633e+08,5.114180e+08,1.363677e+09,3.303010e+11
prev_shareholders_equity,5897.0,8.803351e+08,4.254840e+09,1.630662e+06,7.235638e+07,1.804049e+08,4.896262e+08,1.340870e+11
prev_total_debt,5897.0,1.482403e+09,7.356449e+09,2.989752e+06,1.289376e+08,3.137864e+08,8.450043e+08,2.441621e+11
prev_short_term_debt,5897.0,8.250488e+08,4.298247e+09,1.520389e+06,6.935150e+07,1.684216e+08,4.629355e+08,1.593762e+11
